# 02 — LOB Formation Model

The **LOBFormationMFG** places each agent at a *price distance* x from the
mid-price.  The equilibrium density m*(x) is the limit-order book depth
profile.

### Cost structure

Each agent at position x bears three costs:

| Term | Expression | Economic meaning |
|------|-----------|-----------------|
| Far-field | c_far · x | Cost of posting far from mid (inventory risk) |
| Adverse selection | c_near · e^{−κx} | Sharp execution risk near mid |
| Congestion | c_crowd · m(x) | Cost of competing with other agents |
| Control | c_control · α² | Cost of order repositioning |

The equilibrium balances these: agents neither crowd near the mid (adverse
selection) nor post too far (inventory risk).


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))

import numpy as np
import warnings
warnings.filterwarnings("ignore")

from mfglob.grids import Grid1D, TimeGrid
from mfglob.models.lob_formation import LOBFormationMFG
from mfglob.mfg_solver import MFGSolver


## 1. Solve with default parameters

In [ ]:
model  = LOBFormationMFG(c_far=0.1, c_near=1.0, kappa=2.0,
                        c_crowd=0.3, c_control=0.1, sigma=0.5)
grid   = Grid1D(0.0, 8.0, 80)
tgrid  = TimeGrid(0.5, 60)
solver = MFGSolver(model, grid, tgrid, damping=0.5, tol=1e-3, max_iterations=50)
sol    = solver.solve(verbose=False)

print(f"Converged: {sol['converged']}  ({sol['n_iterations']} iters)")

x     = grid.points
m_T   = sol['density'][-1]
m_avg = np.mean(sol['density'], axis=0)
m_avg /= np.trapezoid(m_avg, x)

mode_idx = np.argmax(m_avg)
print(f"LOB mode (time-averaged): x = {x[mode_idx]:.4f}")
print(f"Near-zero density m(0):   {m_avg[0]:.6f}")


## 2. Steady-state book shape

The time-averaged density m̄(x) gives the equilibrium LOB depth profile.

In [ ]:
print("Equilibrium LOB depth profile (time-averaged):")
print(f"{'x':>8}  {'density':>12}")
print("-" * 24)
idx = list(range(0, len(x), len(x)//12))
for i in idx:
    print(f"{x[i]:>8.4f}  {m_avg[i]:>12.6f}")


## 3. Effect of congestion parameter c_crowd

Higher congestion → agents spread out → shallower peak, heavier tail.

In [ ]:
results = {}
for c_crowd in [0.1, 0.5, 1.0, 2.0]:
    m_loc  = LOBFormationMFG(c_crowd=c_crowd)
    g_loc  = Grid1D(0.0, 8.0, 60)
    tg_loc = TimeGrid(0.5, 40)
    s_loc  = MFGSolver(m_loc, g_loc, tg_loc, damping=0.5,
                       tol=1e-3, max_iterations=40).solve(verbose=False)
    x_l    = g_loc.points
    m_l    = np.mean(s_loc['density'], axis=0)
    m_l   /= np.trapezoid(m_l, x_l)
    mode   = float(x_l[np.argmax(m_l)])
    mean   = float(np.trapezoid(x_l * m_l, x_l))
    results[c_crowd] = {'mode': mode, 'mean': mean}

print(f"{'c_crowd':>10}  {'mode x':>10}  {'mean x':>10}")
print("-" * 34)
for c, r in results.items():
    print(f"{c:>10.2f}  {r['mode']:>10.4f}  {r['mean']:>10.4f}")
print("\nObservation: higher congestion spreads agents further from mid-price.")


## 4. Analytical steady-state comparison

In the limit σ→∞ (diffusion dominates), the steady-state is the Gibbs measure:

$$m^*(x) \propto \exp\!\left(-\frac{c_{\mathrm{far}}\,x + c_{\mathrm{near}}(e^{-\kappa x}-1)}{c_{\mathrm{control}}\,\sigma^2}\right)$$

In [ ]:
x_fine = np.linspace(0.0, 8.0, 500)
c_far, c_near, kappa = 0.1, 1.0, 2.0
c_control, sigma_sq  = 0.1, 0.25

log_m = -(c_far * x_fine + c_near * (np.exp(-kappa * x_fine) - 1.0)) / (c_control * sigma_sq)
log_m -= log_m.max()
m_gibbs = np.exp(log_m)
m_gibbs /= np.trapezoid(m_gibbs, x_fine)

mode_gibbs = x_fine[np.argmax(m_gibbs)]
mean_gibbs = float(np.trapezoid(x_fine * m_gibbs, x_fine))
print(f"Gibbs steady-state: mode={mode_gibbs:.4f}, mean={mean_gibbs:.4f}")
print(f"MFG solution:       mode={x[mode_idx]:.4f}, mean={float(np.trapezoid(x*m_avg,x)):.4f}")
